In [13]:
import sys
sys.path.append('..')
from groq import Groq
import os
from helpers import *
from dotenv import load_dotenv
import json

In [14]:
load_dotenv()

True

In [15]:
client = Groq(api_key=os.getenv("Groq_API_KEY"))

response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {"role": "user", "content": "Say hello in 5 words"}
    ]
)

print(response.choices[0].message.content)

Hello there, nice to meet!


In [16]:


def get_user_intent(user_text):
    prompt = f"""
    User said: "{user_text}"
    
    Extract the following and return ONLY valid JSON:
    
    {{
    "mood": one of [Sad, Tired, Neutral, Happy, Excited, null],
    "genre_preference": genre name or null,
    "reference_title": movie/show name or null,
    "similarity_intent": "similar" or "different" or null,
    "avoid_genres": list of genres to avoid or empty list,
    "vibe": one of [feel-good, suspense, dark, light, emotional, joyful, heartfelt, revenge, inspirational, null]
    }}
    
    Available genres: Action, Adventure, Animation, Children, 
    Comedy, Crime, Documentary, Drama, Fantasy, Film-Noir, 
    Horror, Musical, Mystery, Romance, Sci-Fi, Thriller, War, Western
    
    Note: "rom-com" means Romance + Comedy.
    Only include what's explicitly or implicitly mentioned.
    """
    
    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[{"role": "user", "content": prompt}]
    )
    
    raw = response.choices[0].message.content
    
    try:
        return json.loads(raw)
    except Exception as e:
        print(f"Parse error: {e}")  # shows WHY it failed
        return {
            "mood": None, "genre_preference": None,
            "reference_title": None, "similarity_intent": None,
            "avoid_genres": [],
            "vibe" : None
        }

In [7]:
test_inputs = [
    "had a rough day",
    "I liked Vincenzo, suggest similar ones",
    "watched too many rom-coms, want something different"
]

for text in test_inputs:
    print(f"INPUT: {text}")
    print(f"OUTPUT: {get_user_intent(text)}")
    print("---")

INPUT: had a rough day
OUTPUT: {'mood': 'Sad', 'genre_preference': None, 'reference_title': None, 'similarity_intent': None, 'avoid_genres': [], 'vibe': 'emotional'}
---
INPUT: I liked Vincenzo, suggest similar ones
OUTPUT: {'mood': None, 'genre_preference': None, 'reference_title': 'Vincenzo', 'similarity_intent': 'similar', 'avoid_genres': [], 'vibe': None}
---
INPUT: watched too many rom-coms, want something different
OUTPUT: {'mood': 'Tired', 'genre_preference': None, 'reference_title': None, 'similarity_intent': 'different', 'avoid_genres': ['Romance', 'Comedy'], 'vibe': None}
---


In [8]:
result = get_user_intent("had a rough day")
print(result)
print("Mood:", result["mood"])

{'mood': 'Sad', 'genre_preference': None, 'reference_title': None, 'similarity_intent': None, 'avoid_genres': [], 'vibe': 'emotional'}
Mood: Sad


In [5]:
# For LLM logic (no emojis)
mood_to_genres = {
    "Sad"     : ["Comedy", "Animation", "Musical"],
    "Tired"   : ["Animation", "Comedy", "Children"],
    "Neutral" : ["Action", "Adventure", "Documentary"],
    "Happy"   : ["Comedy", "Romance", "Musical"],
    "Excited" : ["Thriller", "Crime", "Action", "Mystery"]
}

In [3]:
vibe_to_genres = {
    "joyful"        : ["Comedy", "Children", "Musical", "Animation"],
    "suspense"      : ["Thriller", "Mystery", "Crime", "Horror", "Sci-Fi"],
    "revenge"       : ["Action", "Crime", "Thriller", "Drama", "Western"],
    "heartfelt"     : ["Drama", "Romance", "Children", "Adventure"],
    "inspirational" : ["Drama", "Documentary", "Adventure", "War"],
    "feel-good"     : ["Comedy", "Romance", "Children", "Musical", "Adventure"],
    "dark"          : ["Drama", "Thriller", "Crime", "Film-Noir", "Horror", "War", "Sci-Fi"],
    "light"         : ["Comedy", "Children", "Animation", "Musical", "Adventure"],
    "emotional"     : ["Drama", "Romance", "Documentary", "Children"]
}

In [4]:
def resolve_genre(intent, avoided_genres=None):
    if avoided_genres is None:
        avoided_genres = set()

    # 1. Explicit genre — ALWAYS respected, avoid-list ignored 🎯
    if intent.get('genre_preference'):
        return intent['genre_preference']

    # 2. Vibe — pick first NOT avoided
    if intent.get('vibe') and intent['vibe'] in vibe_to_genres:
        for g in vibe_to_genres[intent['vibe']]:
            if g not in avoided_genres:
                return g                      # first survivor wins

    # 3. Mood — same pattern
    if intent.get('mood') and intent['mood'] in mood_to_genres:
        for g in mood_to_genres[intent['mood']]:
            if g not in avoided_genres:
                return g

    # 4. Default
    return "Drama"

In [5]:
# Test: sad mood BUT wants suspense
intent = {
    "genre_preference": None,
    "vibe": "suspense",
    "mood": "Sad"
}
print(resolve_genre(intent))

Thriller


In [13]:
# Test 1 — Sad mood BUT wants suspense
intent = {
    "genre_preference": None,
    "vibe": "suspense",
    "mood": "Sad"
}
print("Sad + suspense →", resolve_genre(intent))
# Should be Thriller (vibe wins!)

Sad + suspense → Thriller


In [14]:
# Test with different intents
test1 = {"genre_preference": "Horror", "mood": None}
print(resolve_genre(test1))  # Horror

test2 = {"genre_preference": None, "mood": "Sad"}
print(resolve_genre(test2))  # Comedy

test3 = {"genre_preference": None, "mood": None}
print(resolve_genre(test3))  # Drama

Horror
Comedy
Drama


In [8]:
def filter_avoid_genres(titles, avoid_genres):
    """
    Remove movies whose genres match 
    any in avoid_genres list.
    """
    if not avoid_genres:
        return titles  # nothing to avoid
    
    filtered = []
    for title in titles:
        # Get this movie's genres
        movie_row = df_movies[df_movies['title'] == title]
        if len(movie_row) == 0:
            continue
        
        movie_genres = movie_row['genres'].values[0]
        
        # Check if ANY avoid genre is in this movie
        should_avoid = False
        for avoid in avoid_genres:
            if avoid in movie_genres:
                should_avoid = True
                break
        
        # Keep only if not avoided
        if not should_avoid:
            filtered.append(title)
    
    return filtered

In [ ]:
def get_smart_recommendations(user_text, user_id=None, n=5):
    '''
    Full LLM pipeline: text → intent → genre → recommendations
    '''
    # Step 1 — Extract intent from text
    intent = get_user_intent(user_text)
   

    # Step 2 — Resolve genre
    genre = resolve_genre(intent)
    

    # Get MORE than needed (buffer for filtering)
    buffer_n = n * 4   # e.g. 20 if n=5
    recommendations = get_recommendations(genre=genre, n=buffer_n, user_id=user_id)
    
    # Filter out avoided genres
    avoid = intent.get('avoid_genres', [])
    recommendations = filter_avoid_genres(recommendations, avoid)
    
    # Return top n from filtered
    return recommendations[:n], genre

In [17]:
# Test each piece separately
text = "had a rough day"

# 1. Intent
intent = get_user_intent(text)
print("1. INTENT:", intent)
print("   Type:", type(intent))

# 2. Genre
genre = resolve_genre(intent)
print("2. GENRE:", genre)
print("   Type:", type(genre))

# 3. Recommendations
recs = get_recommendations(genre=genre, n=5, user_id=None)
print("3. RECS:", recs)
print("   Type:", type(recs))

1. INTENT: {'mood': 'Sad', 'genre_preference': None, 'reference_title': None, 'similarity_intent': None, 'avoid_genres': [], 'vibe': 'emotional'}
   Type: <class 'dict'>
2. GENRE: Drama
   Type: <class 'str'>
3. RECS: ['Ghost and Mrs. Muir, The (1947)', "Razor's Edge, The (1984)", 'Brokeback Mountain (2005)', 'Copycat (1995)', 'Iron Giant, The (1999)']
   Type: <class 'list'>


In [18]:
recs = get_smart_recommendations("had a rough day")
print(recs)

["Bill Burr: I'm Sorry You Feel That Way (2014)", 'Taxi 2 (2000)', 'CHiPS (2017)', 'Little Rascals, The (1994)', 'Love Bug, The (1969)']


In [19]:
test_cases = [
    "had a rough day",
    "I want horror movies",
    "feeling excited and pumped up!",
]

for text in test_cases:
    print(f"\nINPUT: '{text}'")
    recs = get_smart_recommendations(text)
    for r in recs:
        print(f"  → {r}")


INPUT: 'had a rough day'
  → Conversation, The (1974)
  → Science of Sleep, The (La science des rêves) (2006)
  → Good Thief, The (2002)
  → Ro.Go.Pa.G. (1963)
  → Walking and Talking (1996)

INPUT: 'I want horror movies'
  → Confessions (Kokuhaku) (2010)
  → From Dusk Till Dawn (1996)
  → Ghostbusters (2016)
  → Uninvited, The (2009)
  → Frighteners, The (1996)

INPUT: 'feeling excited and pumped up!'
  → A Dog's Purpose (2017)
  → Maiden Heist, The (2009)
  → Happy, Texas (1999)
  → Amazon Women on the Moon (1987)
  → Without a Clue (1988)


In [20]:
# Test avoid_genres
recs = get_smart_recommendations(
    "watched too many rom-coms, want something different"
)
print("Avoiding Romance & Comedy:")
for r in recs:
    print(f"  → {r}")

Avoiding Romance & Comedy:
  → Redline (2009)
  → Dragon Ball: The Path to Power (Doragon bôru: Saikyô e no michi) (1996)
  → Dr. Seuss' The Lorax (2012)
  → Pete's Dragon (1977)


In [21]:
recs = get_smart_recommendations(
    "I liked Vincenzo, suggest similar ones"
)
print("Similar to Vincenzo (Crime):")
for r in recs:
    print(f"  → {r}")

Similar to Vincenzo (Crime):
  → Prime Suspect 6: The Last Witness (2003)
  → Sandpiper, The (1965)
  → Flirting (1991)
  → Tokyo Godfathers (2003)
  → Inside Llewyn Davis (2013)


In [10]:
intent = get_user_intent(
    "I watched Vincenzo, suggest different movies"
)
print(intent)

{'mood': None, 'genre_preference': None, 'reference_title': 'Vincenzo', 'similarity_intent': 'different', 'avoid_genres': [], 'vibe': 'revenge'}


In [23]:
# Check genres of the results
for title in ["With a Friend Like Harry... (Harry, un ami qui vous veut du bien) (2000)",
              "Saragossa Manuscript, The (Rekopis znaleziony w Saragossie) (1965)",
              "Center Stage (2000)"]:
    row = df_movies[df_movies['title'] == title]
    if len(row) > 0:
        print(f"{title[:40]}")
        print(f"   Genres: {row['genres'].values[0]}\n")

With a Friend Like Harry... (Harry, un a
   Genres: Drama|Thriller

Saragossa Manuscript, The (Rekopis znale
   Genres: Adventure|Drama|Mystery

Center Stage (2000)
   Genres: Drama|Musical



In [11]:
intent = get_user_intent(
    "watched too many rom-coms, want something different"
)
print("Intent:", intent)

genre = resolve_genre(intent)
print("Genre:", genre)

# Check BEFORE filtering
raw = get_recommendations(genre=genre, n=15, user_id=None)
print(f"\nBefore filter ({len(raw)} movies):")
for r in raw:
    row = df_movies[df_movies['title'] == r]
    if len(row) > 0:
        print(f"  {r[:35]} → {row['genres'].values[0]}")

Intent: {'mood': None, 'genre_preference': None, 'reference_title': None, 'similarity_intent': 'different', 'avoid_genres': ['Romance', 'Comedy'], 'vibe': None}
Genre: Drama

Before filter (10 movies):
  Mother (Madeo) (2009) → Crime|Drama|Mystery|Thriller
  Ararat (2002) → Drama|War
  All That Heaven Allows (1955) → Drama|Romance
  Beat That My Heart Skipped, The (ba → Drama
  Kundun (1997) → Drama
  Happy Endings (2005) → Comedy|Drama
  The Fundamentals of Caring (2016) → Drama
  Simone (S1m0ne) (2002) → Comedy|Drama|Fantasy|Sci-Fi
  Revenge of the Green Dragons (2014) → Action|Crime|Drama
  Indian Summer (a.k.a. Alive & Kicki → Comedy|Drama


In [ ]:


# Test it!
recs = get_smart_recommendations("had a rough day")
print(recs)

['Shower (Xizao) (1999)', 'Let It Ride (1989)', 'Accidental Tourist, The (1988)', 'Mr. Mom (1983)', 'Bad Milo (Bad Milo!) (2013)']


In [26]:
intent = get_user_intent("feeling tired suggest some feel good movies")
print(intent)

{'mood': 'Tired', 'genre_preference': None, 'reference_title': None, 'similarity_intent': None, 'avoid_genres': [], 'vibe': 'feel-good'}


In [27]:
test_vibes = [
    "I want something that makes me smile",
    "need a good revenge story",
    "something touching and emotional",
    "inspire me with a success story",
    "want a tense suspenseful movie",
]

for text in test_vibes:
    intent = get_user_intent(text)
    print(f"INPUT: {text}")
    print(f"VIBE DETECTED: {intent.get('vibe')}")
    print("---")

INPUT: I want something that makes me smile
VIBE DETECTED: feel-good
---
INPUT: need a good revenge story
VIBE DETECTED: revenge
---
INPUT: something touching and emotional
VIBE DETECTED: emotional
---
INPUT: inspire me with a success story
VIBE DETECTED: inspirational
---
INPUT: want a tense suspenseful movie
VIBE DETECTED: suspense
---


In [28]:
intent = get_user_intent("I want something that makes me smile")
print(intent)  # show the FULL dict

{'mood': None, 'genre_preference': None, 'reference_title': None, 'similarity_intent': None, 'avoid_genres': [], 'vibe': 'joyful'}


In [29]:
# Test 2 — Full pipeline
recs = get_smart_recommendations(
    "feeling sad but want a suspenseful movie"
)
print(recs)

['Thesis (Tesis) (1996)', 'Tourist, The (2010)', 'Horde, The (La Horde) (2009)', 'Scream 4 (2011)', 'By the Gun (2014)']


In [30]:
# Just mood, NO vibe
recs = get_smart_recommendations("had a rough day")
print(recs)
# Should still give Comedy (mood works!)

['Obsession (1965)', 'Day of the Beast, The (Día de la Bestia, El) (1995)', 'Stitch! The Movie (2003)', 'Ant-Man and the Wasp (2018)', 'Kicking and Screaming (1995)']


In [31]:
intent = get_user_intent(
    "had a rough day, suggest some feel good movies"
)
print(intent)

{'mood': 'Sad', 'genre_preference': None, 'reference_title': None, 'similarity_intent': None, 'avoid_genres': [], 'vibe': 'feel-good'}


In [32]:
genre = resolve_genre(intent)
print("Genre chosen:", genre)

Genre chosen: Comedy


In [2]:


# Test vibe beats mood
recs = get_smart_recommendations(
    "feeling sad but want suspense"
)
print(recs)

['Communion (1989)', 'Stray Dog (Nora inu) (1949)', 'Time Lapse (2014)', 'Scream 2 (1997)', 'Suicide Club (Jisatsu saakuru) (2001)']


In [19]:
def generate_explanation(movie, user_context=""):
    """
    Generate a 'Why You'll Like It' reason.
    Accepts EITHER a movie dict (Guest) OR a title string (Personal).
    """
    # Normalize input — figure out title + overview regardless of shape
    if isinstance(movie, dict):
        title = movie.get("title", "")
        overview = movie.get("overview", "")
    else:
        # It's a title string (Personal Mode)
        title = movie
        overview = get_movie_description(title)   # fetch it 🎯

    prompt = f"""
    A user is browsing movie recommendations. {user_context}

    Movie: {title}
    Description: {overview}

    Write ONE short, friendly sentence (max 20 words) explaining why
    they might enjoy this movie. Be specific to the movie. Do NOT
    include the movie title. Return ONLY the sentence, no preamble.
    """

    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content.strip()

In [20]:
# quick test — reuse a movie from discover
movies = discover_movies_by_genre(get_genre_id("Sci-Fi"), n=3)
test_movie = movies[0]

reason = generate_explanation(test_movie, user_context="They're in a Sci-Fi mood.")
print(test_movie["title"], "→", reason)

Disclosure Day → If you love high‑tech conspiracies and mysterious weather clues, this chase‑filled alien mystery will thrill you.


In [21]:
# Pass a TITLE STRING (Personal Mode style), not a dict
reason = generate_explanation("Interstellar (2014)")
print(reason)

If you love breathtaking visuals and emotional journeys through time, this mind‑bending space odyssey will captivate you.
